In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("preprocessed_dataset.csv", encoding="latin-1")

In [3]:
df

,sentiment,clean_text,time_of_tweet,age,log_population,log_land_area,log_density
0,1,i would have responded if i were going,0,0,17.477233,13.389120,4.110874
1,0,soo sad i will miss you here in san diego,1,1,14.872536,10.218335,4.663439
2,0,my boss is bullying me,2,2,17.596309,14.683342,2.944439
3,0,what interview leave me alone,0,3,11.255009,6.154858,5.105945
4,0,sons of why couldn t they put them on the rele...,1,4,17.307958,14.036011,3.295837
...,...,...,...,...,...,...,...
31010,0,its at 3 am im very tired but i cannot sleep b...,1,1,15.706294,11.698085,4.025352
31011,2,all alone in this old house again thanks for t...,2,2,17.002138,14.051926,2.995732
31012,0,i know what you mean my little dog is sinking ...,0,3,19.144064,13.722047,5.424950
31013,2,_sutra what is your next youtube video gonna b...,1,4,17.065064,11.698666,5.370638


In [4]:
X_text = df[["clean_text"]]
y = df["sentiment"]


In [5]:
TEXT_COL = "clean_text"

CAT_COLS = [
    "time_of_tweet",
    "age"
]

NUM_COLS = [
    "log_population",
    "log_land_area",
    "log_density"
]

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer


TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    max_features=50000,
    sublinear_tf=True
)


,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None
,analyzer,'char_wb'
,stop_words,None
,token_pattern,'(?u)\\b\\w\\w+\\b'
,ngram_range,"(3, ...)"


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import classification_report

In [8]:
X = df["clean_text"].astype(str)
y = df["sentiment"]


In [9]:
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True
    )),
    ("clf", LinearSVC(
        class_weight="balanced",
        random_state=42
    ))
])


In [10]:
param_grid = {
    "tfidf__max_features": [20000, 30000, 50000],
    "clf__C": [0.1, 0.5, 1.0, 2.0, 5.0]
}


In [11]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


In [12]:
grid = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)


In [13]:
grid.fit(X, y)


Fitting 5 folds for each of 15 candidates, totalling 75 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'clf__C': [0.1, 0.5, ...], 'tfidf__max_features': [20000, 30000, ...]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [14]:
print("Best F1-macro:", grid.best_score_)
print("Best parameters:", grid.best_params_)


Best F1-macro: 0.7189359526922756
Best parameters: {'clf__C': 0.1, 'tfidf__max_features': 20000}


In [15]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=20000
)

lr_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", LogisticRegression(
        solver="saga",
        max_iter=2000,
        class_weight="balanced",
        n_jobs=-1,
        multi_class="auto"
    ))
])


In [16]:
lr_pipeline.fit(X, y)


C:\Users\ANSHUL MODI\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,steps,"[('tfidf', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,True
,preprocessor,None
,tokenizer,None


In [17]:
lr_param_grid = {
    "tfidf__max_features": [20000, 30000, 50000],
    "clf__C": [0.1, 0.5, 1.0, 2.0, 5.0]
}


In [18]:
lr_grid = GridSearchCV(
    lr_pipeline,
    lr_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

lr_grid.fit(X, y)


Fitting 5 folds for each of 15 candidates, totalling 75 fits


C:\Users\ANSHUL MODI\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


,estimator,Pipeline(step...ver='saga'))])
,param_grid,"{'clf__C': [0.1, 0.5, ...], 'tfidf__max_features': [20000, 30000, ...]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [19]:
print("LR Best F1:", lr_grid.best_score_)
print("LR Best Params:", lr_grid.best_params_)


LR Best F1: 0.7164888869471308
LR Best Params: {'clf__C': 1.0, 'tfidf__max_features': 20000}


In [20]:
from sklearn.linear_model import SGDClassifier

sgd_pipeline = Pipeline([
    ("tfidf", tfidf),
    ("clf", SGDClassifier(
        class_weight="balanced",
        random_state=42
    ))
])


In [21]:
sgd_param_grid = {
    "tfidf__max_features": [20000, 30000, 50000],
    "clf__loss": ["hinge", "log_loss"],  # SVM-like vs LR-like
    "clf__alpha": [1e-4, 5e-4, 1e-3],
    "clf__max_iter": [1000, 2000]
}


In [22]:
sgd_grid = GridSearchCV(
    sgd_pipeline,
    sgd_param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

sgd_grid.fit(X, y)


Fitting 5 folds for each of 36 candidates, totalling 180 fits


,estimator,Pipeline(step...m_state=42))])
,param_grid,"{'clf__alpha': [0.0001, 0.0005, ...], 'clf__loss': ['hinge', 'log_loss'], 'clf__max_iter': [1000, 2000], 'tfidf__max_features': [20000, 30000, ...]}"
,scoring,'f1_macro'
,n_jobs,-1
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [23]:
print("SGD Best F1:", sgd_grid.best_score_)
print("SGD Best Params:", sgd_grid.best_params_)


SGD Best F1: 0.722655844055798
SGD Best Params: {'clf__alpha': 0.0001, 'clf__loss': 'hinge', 'clf__max_iter': 1000, 'tfidf__max_features': 20000}


In [25]:
import pickle

pickle_model_path = "fast_api_model.pkl"
with open(pickle_model_path, "wb") as f:
    pickle.dump(sgd_grid, f)